# 01 - Looking at the raw data

This is the first notebook of the project. Before cleaning anything, the
goal here is just to **look** at the data and understand what condition
it is in.

I use three pandas functions on every table:
- `head()` to see some rows
- `info()` to see the columns, their type, and how many cells are not empty
- `isna().sum()` to count missing values in each column


In [ ]:
import pandas as pd

RAW = "../../datasets/raw"
DIM = "../../datasets/dim"


## A quick look before anything else

Before checking each table, I open the production one and see how it
looks. Real data always comes with a few problems -- that's normal.


In [ ]:
df_production = pd.read_csv(f"{RAW}/fact_production_raw.csv", encoding="utf-8-sig")
print(df_production.head(10))


In [ ]:
print(df_production["Process"].unique())


Notice: `Injection Molding` and `injection molding` show up as if they
were different processes, but they're the same thing written
differently. This is a classic real-data problem -- I fix it in the
next notebook (`02_data_cleaning_production.ipynb`) with
`.str.strip().str.title()`.


## 1. Production tables

Now I repeat the three commands (`head`, `info`, `isna().sum()`) for
each of the four production tables. Instead of copying the code four
times, I use a list of file names and a `for` loop.


In [ ]:
production_files = [
    "fact_production_plan_raw.csv",
    "fact_production_raw.csv",
    "fact_downtime_raw.csv",
    "fact_material_consumption_raw.csv",
]

for file_name in production_files:
    path = f"{RAW}/{file_name}"
    df = pd.read_csv(path, encoding="utf-8-sig")

    print("=" * 70)
    print("Table:", file_name)
    print("Rows and columns:", df.shape)

    print("\nFirst rows:")
    print(df.head(5))

    print("\nInfo (columns, types, non-null):")
    print(df.info())

    print("\nMissing values per column:")
    print(df.isna().sum())


## 2. Quality control tables

These are the 8 tables from the quality lab: measurements, inspections,
and lot approval/rejection decisions for caps, bottles, and screen
printing ink.


In [ ]:
quality_files = [
    "fact_cap_inspection_variable_cq_raw.csv",
    "fact_cap_attribute_inspection_cq_raw.csv",
    "fact_cap_disposition_lot_cq_raw.csv",
    "fact_bottle_inspection_variables_cq_raw.csv",
    "fact_bottle_attribute_inspection_cq_raw.csv",
    "fact_bottle_disposition_lot_cq_raw.csv",
    "fact_ink_attribute_inspection_cq_raw.csv",
    "fact_ink_disposition_lot_cq_raw.csv",
]

for file_name in quality_files:
    path = f"{RAW}/{file_name}"
    df = pd.read_csv(path, encoding="utf-8-sig")

    total_missing = df.isna().sum().sum()

    print("=" * 70)
    print("Table:", file_name)
    print("Rows and columns:", df.shape)
    print("Total missing values:", total_missing)
    print(df.head(5))


## 3. Dimension tables (reference data)

These tables come from engineering (machine capacities, cap/bottle
specs, control plans), not from the shop floor -- so I expect them to
be cleaner than the production/quality tables. Let's check.


In [ ]:
dimension_files = [
    "dim_masterbatch.csv",
    "dim_machine_setup.csv",
    "dim_cap.csv",
    "dim_cap_control_plan_cq.csv",
    "dim_bottle_control_plan_cq.csv",
    "dim_ink_control_plan_cq.csv",
    "dim_raw_material_control_plan.csv",
    "dim_customer.csv",
    "dim_supplier.csv",
]

for file_name in dimension_files:
    path = f"{DIM}/{file_name}"
    df = pd.read_csv(path, encoding="utf-8-sig")

    print("=" * 70)
    print("Table:", file_name)
    print("Rows and columns:", df.shape)
    print("Total missing values:", df.isna().sum().sum())


## 4. What I found

Looking at the results above, these are the problems I'll need to fix
in the next notebooks:

- **Categories written in different ways**, like `Injection Molding`
  and `injection molding` -- fixed with `.str.strip().str.title()`.
- **Missing values** in a few columns, mostly `OperatorId` and
  `MaintenanceTechnician`.
- **Negative numbers** where they shouldn't be, like `RejectedQty`
  (there's no such thing as rejecting a negative quantity of pieces).
- **Duplicate rows** in a few tables -- fixed with `drop_duplicates()`.
- Some "empty cells" actually have a space `" "` or a dash `"-"`
  typed in, instead of being truly empty. `isna()` doesn't catch these,
  so I need to handle that separately in the next notebook.
- The dimension tables really do arrive cleaner, as expected.

Cleaning and organizing data like this is usually the biggest chunk of
a data project -- exactly what the next notebook,
`02_data_cleaning_production.ipynb`, does.
